# PILOT — Claude Sonnet 5.5 — 5-shot Experiment B

**Run this before the full notebook** (`Sonnet_ExperimentB.ipynb`). It makes ~25 real, synchronous calls (condition `fs_B`), checks image handling, prompt format, answer parsing and token usage, and prints a GO / NO-GO verdict with the projected cost of the full 1600-image run.

* Costs a few cents. Nothing is written to your results files.
* Pilot answers are saved to Drive and are not paid for twice if you re-run the cell.
* Add `ANTHROPIC_API_KEY` in Colab **Secrets** first.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

## 2. Results folder (pilot answers are cached here)

In [ ]:
import os

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

## 3. Install packages

In [ ]:
!pip install -q -U anthropic

## 4. Connect to Claude Sonnet 5.5 & configuration
Everything that affects the results is set here. `temperature` / `top_p` / `top_k` are **not** sent (Claude Sonnet 5.5 rejects non-default sampling parameters), so runs are not strictly deterministic. Thinking is adaptive and **effort is `medium`**, set explicitly so that all Claude models run at the same effort (Sonnet 5.5's own default is `high`).
Prices are Anthropic's published prices for Claude Sonnet 5.5 ($2 / $10 per 1M tokens) at the Batch discount — check them on Anthropic's pricing page. `QUEUE_REQUEST_LIMIT` is the number of batch requests that may wait in the queue at once (Start tier 200,000 / Build 300,000 / Scale 500,000), so it never binds for a 1,600-request experiment.

In [ ]:
import os, io, re, json, time, math, base64, random, hashlib
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=5)

COND = "fs_B"
MODEL      = "claude-sonnet-5-5"
EFFORT     = "medium"            # low | medium | high | xhigh | max   (set explicitly; Sonnet 5.5 default = high)
MAX_TOKENS = 16000               # cap incl. thinking tokens (not a cost)

PRICE_IN, PRICE_OUT = 2.00, 10.00 # USD per 1M tokens (standard price) -- VERIFY
BATCH_DISCOUNT = 0.5             # Message Batches API = 50% of standard prices
N_PILOT, PILOT_SEED, PILOT_WORKERS = 25, 42, 4

MAX_BATCH_MB        = 200        # hard limit is 256 MB per batch
QUEUE_REQUEST_LIMIT = 200_000    # batch requests allowed in the processing queue at once (see the note above)

CONFIG = dict(model=MODEL, effort=EFFORT, thinking="adaptive", max_tokens=MAX_TOKENS, image_max=512, cond=COND)
CONFIG_HASH = hashlib.sha1(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
print(CONFIG, CONFIG_HASH)

## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Same resize as the open-source runs."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

@lru_cache(maxsize=3000)
def img_b64(path):
    buf = io.BytesIO()
    load_image(path).save(buf, format="JPEG", quality=95)
    return base64.standard_b64encode(buf.getvalue()).decode()

def _img(path):
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64(path)}}

## 6. Prompt builder — Experiment B
Text is identical to the 5-shot prompt of the open-source notebooks. The builder returns one Messages API request body (the same body is used for the pilot and for the Batch file).

Experiment B differs from A **only** in the YOLO block of the final turn.

In [ ]:
SYSTEM_PROMPT = """You are an expert in crowd counting and crowd density estimation.

You will first receive five labelled support examples.

Each support example contains:
- one image
- the exact crowd count
- the correct crowd density

Learn from these examples.

Then you will receive one test image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""

SUPPORT_TURN = """Support Example {i}

Study this labelled support example carefully.
"""

SUPPORT_ANSWER = """Ground Truth

Count: {count}

Density: {density} ({label})
"""

TEST_TURN = """TEST IMAGE

YOLO predicted count: {yolo}

Use the YOLO predicted count as additional information, but inspect the image yourself.

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""


def _params(system, messages):
    """One Messages-API request body. It is used as-is for the pilot and as `params` of a Batch request."""
    return {"model": MODEL, "max_tokens": MAX_TOKENS, "system": system, "messages": messages,
            "thinking": {"type": "adaptive"}, "output_config": {"effort": EFFORT}}


def build_messages_B(sample, support_images, test_image, yolo_count):
    """Messages-API request body for one test image — 5-shot Experiment B."""
    msgs = []
    for i in range(1, 6):
        count = int(sample[f"support{i}_count"])
        density = int(sample[f"support{i}_density"])
        msgs.append({"role": "user", "content": [
            _img(support_images[i - 1]),
            {"type": "text", "text": SUPPORT_TURN.format(i=i)}]})
        msgs.append({"role": "assistant",
                     "content": SUPPORT_ANSWER.format(count=count, density=density, label=DENSITY_TEXT[density])})
    msgs.append({"role": "user", "content": [_img(test_image), {"type": "text", "text": TEST_TURN.format(yolo=int(yolo_count))}]})
    return _params(SYSTEM_PROMPT, msgs)


## 7. Inference and parsing (same robust parser used throughout this project)

In [ ]:
def run_claude(request, retries=3):
    """One synchronous call (pilot / 9c re-run only). Returns (message_dict, error_string)."""
    last = None
    for attempt in range(retries):
        try:
            return client.messages.create(**request).model_dump(mode="json"), None
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
    return None, last


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than taking them literally."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

def extract_text(msg):
    """Answer text = all `text` blocks. `thinking` blocks (empty by default) are ignored."""
    if not msg:
        return ""
    return "".join(b.get("text", "") for b in (msg.get("content") or []) if b.get("type") == "text").strip()


def usage_of(msg):
    """in_tok = input tokens; out_tok = output tokens INCLUDING thinking (the API does not split them)."""
    u = (msg or {}).get("usage") or {}
    return dict(in_tok=(u.get("input_tokens") or 0) + (u.get("cache_creation_input_tokens") or 0) + (u.get("cache_read_input_tokens") or 0),
                out_tok=u.get("output_tokens") or 0)


def row_from_message(fname, msg, err=None):
    """One result row from a Messages-API message dict (used for the pilot and when assembling the batch results)."""
    text = extract_text(msg)
    pc, pd_ = parse_response(text)
    stop = (msg or {}).get("stop_reason")
    return dict(test_filename=int(fname), predicted_count=pc, predicted_density=pd_, raw_response=text,
                status=stop, incomplete=(stop if stop not in (None, "end_turn", "stop_sequence") else None),
                error=err, **usage_of(msg))


# offline self-test (no API call)
_fake = {"stop_reason": "end_turn", "content": [{"type": "thinking", "thinking": "", "signature": "x"},
         {"type": "text", "text": "Count: 123\n\nDensity: 2"}],
         "usage": {"input_tokens": 900, "output_tokens": 400}}
assert parse_response(extract_text(_fake)) == (123, 2)
assert parse_response("Count: 12345678\nDensity: 1") == (None, 1)
assert usage_of(_fake) == dict(in_tok=900, out_tok=400)
assert row_from_message(7, _fake)["incomplete"] is None
assert row_from_message(7, dict(_fake, stop_reason="max_tokens"))["incomplete"] == "max_tokens"
print("Parser self-test passed.")

## 8. Load data & smoke test (25-image pilot) — READ THIS OUTPUT BEFORE RUNNING THE FULL LOOP
Prints the rendered prompt structure, runs 25 random test images with real calls, and reports parse rate, tokens per request (and per image), MAE and the projected full cost with a GO / NO-GO verdict.

In [ ]:
import pandas as pd
import os

ROOT = "/content/Phase5_Input"
TRAIN_DIR = f"{ROOT}/train/images"
TEST_DIR = f"{ROOT}/test/images"

support_df = pd.read_csv(f"{ROOT}/support_mapping.csv")
test_df = pd.read_csv(f"{ROOT}/test.csv")
yolo_df = pd.read_csv(f"{ROOT}/yolo_predictions.csv")
YOLO = dict(zip(yolo_df["image_id"].astype(int), yolo_df["predicted_count"].astype(int)))

SUPPORT = {int(r["test_filename"]): r for _, r in support_df.iterrows()}
TEST_FILES = [int(f) for f in support_df["test_filename"]]      # same order as the open-source 5-shot runs

def test_path(f):    return os.path.join(TEST_DIR, f"{int(f):04d}.jpg")
def support_paths(f): return [os.path.join(TRAIN_DIR, f"{int(SUPPORT[int(f)][f'support{i}_filename']):04d}.jpg") for i in range(1, 6)]

def make_request(f):
    f = int(f)
    return build_messages_B(SUPPORT[f], support_paths(f), test_path(f), YOLO[f])

# ---- data checks ----
assert len(TEST_FILES) == len(set(TEST_FILES)) == 1600, len(TEST_FILES)
assert set(TEST_FILES) == set(test_df["filename"].astype(int)), "support_mapping and test.csv differ"
assert all(f in YOLO for f in TEST_FILES), "missing YOLO counts"
_missing = [p for f in TEST_FILES for p in [test_path(f)] + support_paths(f) if not os.path.exists(p)]
assert not _missing, f"{len(_missing)} image files missing, e.g. {_missing[:3]}"
print("Data checks passed:", len(TEST_FILES), "test images")

In [ ]:
def render(req):
    out = ["[system] " + req["system"].split("\n")[0] + " ..."]
    for m in req["messages"]:
        if isinstance(m["content"], str):
            out.append(f"[{m['role']}] {m['content']!r}")
        else:
            parts = ["<image>" if c["type"] == "image" else c["text"] for c in m["content"]]
            out.append(f"[{m['role']}] " + " | ".join(repr(p) for p in parts))
    return "\n".join(out)

_req = make_request(TEST_FILES[0])
_n_img = sum(1 for m in _req["messages"] if not isinstance(m["content"], str) for c in m["content"] if c["type"] == "image")
assert _n_img == 6, _n_img
assert not ({"temperature", "top_p", "top_k", "stream"} & set(_req))
assert _req["thinking"] == {"type": "adaptive"} and _req["output_config"] == {"effort": EFFORT}
assert _req["messages"][-1]["role"] == "user" and all(m["role"] in ("user", "assistant") for m in _req["messages"])
assert ("YOLO predicted count: " in render(_req)) == True
print(render(_req))
print("\nImages in request:", _n_img, "| request size:", round(len(json.dumps(_req)) / 1024), "KB")

In [ ]:
PILOT_PATH = f"{RESULT_DIR}/sonnet_pilot_{COND}_{CONFIG_HASH}.jsonl"
PILOT_FILES = sorted(random.Random(PILOT_SEED).sample(TEST_FILES, N_PILOT))

pilot = {}
if os.path.exists(PILOT_PATH):
    for line in open(PILOT_PATH, encoding="utf-8"):
        d = json.loads(line); pilot[d["fname"]] = d
print("Resumed from disk:", len(pilot))

def _one(f):
    msg, err = run_claude(make_request(f))
    return {"fname": int(f), "msg": msg, "err": err}

todo = [f for f in PILOT_FILES if f not in pilot]
with open(PILOT_PATH, "a", encoding="utf-8") as out, ThreadPoolExecutor(PILOT_WORKERS) as ex:
    for d in ex.map(_one, todo):
        pilot[d["fname"]] = d
        out.write(json.dumps(d) + "\n"); out.flush()
print("Pilot calls stored:", len(pilot))

P = pd.DataFrame([row_from_message(d["fname"], d["msg"], d["err"]) for d in pilot.values()])
gt = test_df.set_index("filename")
P["true_count"] = P["test_filename"].map(gt["total_count"])
P["true_density"] = P["test_filename"].map(gt["crowd_density"])
P["cost_batch"] = (P.in_tok * PRICE_IN + P.out_tok * PRICE_OUT) / 1e6 * BATCH_DISCOUNT

ok = P.dropna(subset=["predicted_count"])
print(f"\nPilot ({len(P)} images)  [{COND}]")
print("  count parsed   :", f"{100 * P.predicted_count.notna().mean():.0f}%")
print("  density parsed :", f"{100 * P.predicted_density.notna().mean():.0f}%")
print("  API errors     :", int(P.error.notna().sum()), "| incomplete (max_tokens / refusal):", int(P.incomplete.notna().sum()), "| stop reasons:", P.status.value_counts().to_dict())
print("  tokens/request : in", round(P.in_tok.mean()), "| out (incl. thinking)", round(P.out_tok.mean()), "| max out", int(P.out_tok.max()))
print("  pilot MAE      :", round((ok.predicted_count - ok.true_count).abs().mean(), 1), "| density acc:", f"{100 * (P.predicted_density == P.true_density).mean():.0f}%")

print("  input tokens per image (approx):", round(P.in_tok.mean() / _n_img), " <- a 512x384 image should be roughly 260 tokens by Anthropic's width x height / 750 rule (zero-shot requests also include the ~250-token prompt)")
proj = P.cost_batch.mean() * len(TEST_FILES)
print(f"  projected full run: ~${proj:.2f} on Batch (~${proj / BATCH_DISCOUNT:.2f} standard)")

problems = []
if P.error.notna().any(): problems.append("API errors")
if P.incomplete.notna().any(): problems.append("incomplete responses -> raise MAX_TOKENS (or a refusal: read raw_response / stop reason)")
if P.predicted_count.isna().mean() > 0.02: problems.append("count parse rate < 98%")
if P.predicted_density.isna().mean() > 0.02: problems.append("density parse rate < 98%")
print("\nVERDICT:", "NO-GO -> " + "; ".join(problems) if problems else "GO (no blocking problems in the pilot)")
print(P[["test_filename", "true_count", "predicted_count", "true_density", "predicted_density", "out_tok", "raw_response"]].head(5).to_string())

In [ ]:
P.to_csv(f"{RESULT_DIR}/sonnet_pilot_report_{COND}.csv", index=False)
print("Saved pilot report:", f"{RESULT_DIR}/sonnet_pilot_report_{COND}.csv")

## Next step
If the verdict above is **GO**, open `Sonnet_ExperimentB.ipynb`, and run it top to bottom. Set `CONFIRM_FULL_RUN = True` in section 9a-2 only when you are ready to spend the projected amount.
If **NO-GO**, fix what the verdict lists (for example raise `MAX_TOKENS` for incomplete responses) and re-run the pilot.